# Lightweight SED phase and orientation mapping

**Purpose.** Preprocess a perovskite SED/4D-STEM scan, obtain a simple
unsupervised segmentation using K-means, and map crystal phase and
orientation by matching experimental diffraction patterns to simulated
α-FAPbI₃ and δ-FAPbI₃ templates.

**Input.** A calibrated HyperSpy/pyxem diffraction signal and CIF files
for the candidate phases.

**Output.** Cluster-average diffraction patterns and an orix
`CrystalMap` containing phase/orientation assignments.

K-means is used here as an accessible exploratory segmentation for simple
datasets. It is not the SIGMA machine-learning pipeline and its clusters
should not be interpreted as phases without diffraction-based validation.

## 1. Setup and parameters

All data paths and experiment-specific values are defined here. The
notebook does not contain machine-specific absolute paths and does not
delete existing output directories.

In [ ]:
%matplotlib inline

from pathlib import Path

import hyperspy.api as hs
import matplotlib.pyplot as plt
import numpy as np
import pyxem as pxm
from diffsims.generators.simulation_generator import SimulationGenerator
from orix import io, plot
from orix.crystal_map import Phase
from orix.quaternion import Orientation
from orix.sampling import get_sample_reduced_fundamental
from orix.vector import Miller, Vector3d
from sklearn.cluster import KMeans

hs.set_log_level("ERROR")
print("HyperSpy:", hs.__version__)
print("pyxem:", pxm.__version__)

In [ ]:
# Repository-relative paths. This works when Jupyter starts from either
# the repository root or its notebooks/ directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_ROOT / "data/perovskite_sed.zspy"
ALPHA_CIF_PATH = REPO_ROOT / "structures/FAPbI3_alpha_cubic.cif"
DELTA_CIF_PATH = REPO_ROOT / "structures/FAPbI3_delta_2H.cif"
OUTPUT_DIR = REPO_ROOT / "outputs/orientation_mapping"

# Acquisition calibration
NAVIGATION_SCALE_NM = (4.55, 4.55)
DIFFRACTION_CALIBRATION_INV_A_PER_PX = 0.0132885495311464
ACCELERATING_VOLTAGE_KV = 300

# Scan preprocessing
NAVIGATION_Y_STOP = 155       # Set to None to retain the full scan.
DIRECT_BEAM_METHOD = "blur"
DIRECT_BEAM_SIGMA_PX = 5
DIRECT_BEAM_HALF_WIDTH_PX = 20
NAVIGATION_REBIN = (2, 2, 1, 1)

# Cartesian diffraction-background subtraction
BACKGROUND_METHOD = "difference of gaussians"  # or "h-dome"
DOG_MIN_SIGMA = 1.5
DOG_MAX_SIGMA = 5.0
H_DOME_HEIGHT = 0.15

# Deterministic K-means segmentation
N_CLUSTERS = 5
KMEANS_N_INIT = 25
KMEANS_RANDOM_STATE = 0
CLUSTER_NAVIGATION_REBIN = (3, 3, 1, 1)

# Diffraction simulation and orientation sampling
ALPHA_ANGULAR_RESOLUTION_DEG = 1
DELTA_ANGULAR_RESOLUTION_DEG = 3
RECIPROCAL_RADIUS_INV_A = 1.16
MAX_EXCITATION_ERROR_INV_A = 0.04
MINIMUM_SIMULATED_INTENSITY = 1e-3
PRECESSION_ANGLE_DEG = 0

# Polar transform and matching
POLAR_RADIAL_BINS = 64
POLAR_AZIMUTHAL_BINS = 360
POLAR_BACKGROUND_PERCENTILE = 80
INTENSITY_POWER = 0.2
TEMPLATE_FRACTION_TO_KEEP = 1.0
N_BEST_MATCHES = 1

# One position for reporting a representative α-phase zone axis
ZONE_AXIS_POSITION = (80, 41)

## 2. Load, calibrate and crop the diffraction signal

The public workflow starts from `.zspy` or another HyperSpy-readable
diffraction file. Instrument-specific Merlin `.mib` conversion is outside
the scope of this repository.

In [ ]:
required_inputs = [DATA_PATH, ALPHA_CIF_PATH, DELTA_CIF_PATH]
missing = [path for path in required_inputs if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required input(s): "
        + ", ".join(str(path) for path in missing)
        + ". Update the paths in the parameter cell."
    )

signal = hs.load(DATA_PATH, lazy=True)
print(signal)
print(signal.axes_manager)

signal.axes_manager[0].name = "x"
signal.axes_manager[0].scale = NAVIGATION_SCALE_NM[0]
signal.axes_manager[0].units = "nm"
signal.axes_manager[0].offset = 0

signal.axes_manager[1].name = "y"
signal.axes_manager[1].scale = NAVIGATION_SCALE_NM[1]
signal.axes_manager[1].units = "nm"
signal.axes_manager[1].offset = 0

signal.set_diffraction_calibration(DIFFRACTION_CALIBRATION_INV_A_PER_PX)

processed_signal = (
    signal.inav[:, :NAVIGATION_Y_STOP]
    if NAVIGATION_Y_STOP is not None
    else signal
)
print(processed_signal.axes_manager)

## 3. Centre and rebin the scan

Direct-beam shifts are estimated across navigation space and reduced to a
linear plane before centring. Navigation rebinning improves signal to
noise without changing diffraction-space sampling.

In [ ]:
beam_shifts = processed_signal.get_direct_beam_position(
    method=DIRECT_BEAM_METHOD,
    sigma=DIRECT_BEAM_SIGMA_PX,
    half_square_width=DIRECT_BEAM_HALF_WIDTH_PX,
)
if hasattr(beam_shifts.data, "compute"):
    beam_shifts.compute()

beam_shift_plane = beam_shifts.get_linear_plane()
processed_signal.center_direct_beam(
    shifts=beam_shift_plane,
    inplace=True,
)

if NAVIGATION_REBIN != (1, 1, 1, 1):
    processed_signal = processed_signal.rebin(scale=NAVIGATION_REBIN)

processed_signal.plot(cmap="viridis", gamma=0.5, scalebar_color="k")

## 4. Subtract diffraction background

The original exploration compared several methods. The public notebook
exposes one Cartesian method at a time through `BACKGROUND_METHOD`.

In [ ]:
if BACKGROUND_METHOD == "difference of gaussians":
    filtered_signal = processed_signal.subtract_diffraction_background(
        "difference of gaussians",
        inplace=False,
        min_sigma=DOG_MIN_SIGMA,
        max_sigma=DOG_MAX_SIGMA,
    )
elif BACKGROUND_METHOD == "h-dome":
    filtered_signal = processed_signal.subtract_diffraction_background(
        "h-dome",
        inplace=False,
        h=H_DOME_HEIGHT,
    )
else:
    raise ValueError(
        "BACKGROUND_METHOD must be 'difference of gaussians' or 'h-dome'."
    )

filtered_signal.plot(cmap="viridis")

## 5. Segment diffraction behaviour with deterministic K-means

`random_state` fixes centroid initialization, making repeated runs
reproducible for unchanged input data and parameters. Cluster-average
patterns are saved without clearing or deleting existing directories.

In [ ]:
clustering_signal = filtered_signal
if CLUSTER_NAVIGATION_REBIN != (1, 1, 1, 1):
    clustering_signal = clustering_signal.rebin(
        scale=CLUSTER_NAVIGATION_REBIN
    )

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    n_init=KMEANS_N_INIT,
    random_state=KMEANS_RANDOM_STATE,
)
clustering_signal.cluster_analysis(
    cluster_source="signal",
    algorithm=kmeans,
)

clustering_signal.plot_cluster_results()
clustering_signal.plot_cluster_signals(signal="sum")

In [ ]:
cluster_output_dir = OUTPUT_DIR / "cluster_patterns"
cluster_output_dir.mkdir(parents=True, exist_ok=True)

cluster_patterns = clustering_signal.get_cluster_signals(signal="sum")
for cluster_index in range(len(cluster_patterns)):
    destination = cluster_output_dir / f"cluster_{cluster_index:02d}.txt"
    np.savetxt(destination, cluster_patterns.inav[cluster_index].data)
    print("Saved:", destination)

## 6. Build crystallographic template banks

Cubic α-FAPbI₃ and 2H hexagonal δ-FAPbI₃ are sampled over their reduced
orientation spaces. The α phase uses a finer grid because it is the target
long-range ordered phase in the stabilized film.

In [ ]:
alpha_phase = Phase.from_cif(ALPHA_CIF_PATH)
alpha_phase.name = "Alpha_cubic"
alpha_phase.color = "tab:blue"
alpha_rotations = get_sample_reduced_fundamental(
    ALPHA_ANGULAR_RESOLUTION_DEG,
    point_group=alpha_phase.point_group,
)

delta_phase = Phase.from_cif(DELTA_CIF_PATH)
delta_phase.name = "Delta_2Hexagonal"
delta_phase.color = "tab:orange"
delta_rotations = get_sample_reduced_fundamental(
    DELTA_ANGULAR_RESOLUTION_DEG,
    point_group=delta_phase.point_group,
)

print("α templates:", alpha_rotations.size)
print("δ templates:", delta_rotations.size)

In [ ]:
simulation_generator = SimulationGenerator(
    accelerating_voltage=ACCELERATING_VOLTAGE_KV,
    minimum_intensity=MINIMUM_SIMULATED_INTENSITY,
    precession_angle=PRECESSION_ANGLE_DEG,
    approximate_precession=True,
)

simulations = simulation_generator.calculate_diffraction2d(
    phase=[alpha_phase, delta_phase],
    rotation=[alpha_rotations, delta_rotations],
    reciprocal_radius=RECIPROCAL_RADIUS_INV_A,
    with_direct_beam=False,
    max_excitation_error=MAX_EXCITATION_ERROR_INV_A,
)
simulations.plot(size_factor=1)

## 7. Transform to polar coordinates and match orientations

A radial-percentile background is subtracted in polar space. pyxem then
screens and correlates experimental patterns against the simulated bank,
retaining the best phase/orientation assignment at each navigation pixel.

In [ ]:
polar_signal = filtered_signal.get_azimuthal_integral2d(
    npt=POLAR_RADIAL_BINS,
    npt_azim=POLAR_AZIMUTHAL_BINS,
    inplace=False,
    mean=False,
)
polar_signal = polar_signal.subtract_diffraction_background(
    "radial percentile",
    percentile=POLAR_BACKGROUND_PERCENTILE,
    inplace=False,
)
polar_signal = polar_signal ** INTENSITY_POWER
polar_signal.plot(cmap="magma_r")

In [ ]:
orientation_map = polar_signal.get_orientation(
    simulations,
    frac_keep=TEMPLATE_FRACTION_TO_KEEP,
    n_best=N_BEST_MATCHES,
    normalize_templates=True,
)

orientation_map.plot_over_signal(
    filtered_signal,
    cmap="magma_r",
    gamma=INTENSITY_POWER,
)

## 8. Create and export phase/orientation maps

IPF colours are calculated independently for the α and δ phase subsets.
The `CrystalMap` is the reusable machine-readable output.

In [ ]:
crystal_map = orientation_map.to_crystal_map()
crystal_map.plot()

alpha_key = plot.IPFColorKeyTSL(
    alpha_phase.point_group,
    direction=Vector3d.zvector(),
)
delta_key = plot.IPFColorKeyTSL(
    delta_phase.point_group,
    direction=Vector3d.zvector(),
)

alpha_orientations = crystal_map["Alpha_cubic"].orientations
delta_orientations = crystal_map["Delta_2Hexagonal"].orientations
alpha_colours = alpha_key.orientation2color(alpha_orientations)
delta_colours = delta_key.orientation2color(delta_orientations)

crystal_map["Alpha_cubic"].plot(alpha_colours[:, 0, :])
crystal_map["Delta_2Hexagonal"].plot(delta_colours[:, 0, :])

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
crystal_map_path = OUTPUT_DIR / "phase_orientation_map.h5"
io.save(filename=crystal_map_path, object2write=crystal_map)
print("CrystalMap written to:", crystal_map_path)

## 9. Report a representative zone axis

This optional check assumes the selected point belongs to the cubic α
phase. Change `ZONE_AXIS_POSITION` to a validated α-phase pixel.

In [ ]:
navigation_shape = filtered_signal.data.shape[:2]
zone_x = min(max(ZONE_AXIS_POSITION[0], 0), navigation_shape[0] - 1)
zone_y = min(max(ZONE_AXIS_POSITION[1], 0), navigation_shape[1] - 1)

representative_rotation = orientation_map.inav[zone_x, zone_y].to_rotation()
beam_vector = (
    Orientation(representative_rotation, alpha_phase.point_group)
    * Vector3d.zvector()
)
zone_axis = Miller(xyz=beam_vector.data, phase=alpha_phase)
zone_axis.coordinate_format = "hkl"
print("Representative α-phase zone axis:", zone_axis.round())

## Interpretation and limitations

Validate phase assignments against the cluster-average patterns, simulated
overlays and known specimen geometry. Results depend on diffraction
calibration, background subtraction, candidate structures, orientation
sampling and correlation thresholds. K-means clusters describe similarity
in the selected signal representation; they do not independently establish
composition, phase or crystallographic orientation.